# Câu b: Tìm tập phổ biến bằng FP-Growth và ghi nhận thời gian xử lý

Phần 2 – Bài 6 (trang 45): Groceries – 9.835 giao dịch, 169 mặt hàng.

**Về khoảng min_sup 0.01% – 0.1%:** với 9.835 giao dịch, min_sup = 0.01% tương đương support count ≥ 1, tức là *mọi* tập con của *mọi* giao dịch đều phổ biến (giao dịch dài nhất có 32 mặt hàng → tới 2³² tập con), nên không thuật toán nào chạy hết được. Thử thực tế trên máy 8 GB RAM: Apriori bị tràn bộ nhớ từ min_sup = 0.04% trở xuống. Vì vậy bài này khảo sát **min_sup từ 0.1% giảm dần tới 0.05%** (bước 0.01%) cho cả hai thuật toán, và chạy thêm FP-Growth ở 0.04%, 0.03%, 0.02% để thấy FP-Growth vẫn xử lý được khi Apriori đã không chạy nổi.

In [1]:
SV = "Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT"
DATA = "../../data/"  # thư mục chứa dữ liệu
print(SV)

Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT


In [2]:
import time
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth

---
## Thực hiện yêu cầu

In [3]:
print(SV)
# Mỗi dòng là một giao dịch, các mặt hàng cách nhau bởi dấu phẩy (số mặt hàng mỗi dòng khác nhau)
with open(DATA + 'groceries.csv', encoding='utf-8') as f:
    transactions = [line.strip().split(',') for line in f if line.strip()]

te = TransactionEncoder()
df = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)

SUPPORTS = [0.001, 0.0009, 0.0008, 0.0007, 0.0006, 0.0005]   # 0.1% -> 0.05%
SUPPORTS_FP_EXTRA = [0.0004, 0.0003, 0.0002]                 # chỉ FP-Growth chạy được
print('Số giao dịch:', len(transactions))

Họ tên: Mai Quốc Đại - MSSV: 6451071013 - Lớp: CQ.64.CNTT
Số giao dịch: 9835


In [4]:
res_fp = []
for s in SUPPORTS + SUPPORTS_FP_EXTRA:
    t0 = time.perf_counter()
    fi = fpgrowth(df, min_support=s, use_colnames=True)
    res_fp.append({'min_sup (%)': s * 100, 'số tập phổ biến': len(fi), 'thời gian FP-Growth (s)': round(time.perf_counter() - t0, 3)})
res_fp = pd.DataFrame(res_fp)
res_fp

,min_sup (%),số tập phổ biến,thời gian FP-Growth (s)
0,0.10,13492,0.475
1,0.09,16188,0.506
2,0.08,19979,0.595
3,0.07,25570,0.654
4,0.06,34353,0.792
5,0.05,49580,0.968
6,0.04,80388,1.367
7,0.03,166682,2.041
8,0.02,790072,6.298


**Nhận xét:**

- FP-Growth tìm được **đúng cùng số tập phổ biến** như Apriori ở mọi mức min_sup từ 0,1% đến 0,05%.
- Thời gian FP-Growth chỉ khoảng 0,5–1 giây và tăng chậm khi giảm min_sup.
- FP-Growth vẫn chạy được ở các mức nhỏ hơn mà Apriori không chạy nổi: 0,04% (≈80 nghìn tập), 0,03% (≈167 nghìn tập), 0,02% (≈790 nghìn tập) chỉ trong vài giây.
- Ở 0,02% số tập phổ biến đã tăng gần 60 lần so với 0,1%; xuống 0,01% (support count ≥ 1) thì mọi tập con của mọi giao dịch đều phổ biến, nên không thể liệt kê hết.